In [ ]:
# 1-الف
import sys
from operator import add
from pyspark.sql import SparkSession
import time
startTime = time.time()

if __name__ == "__main__":

    spark = SparkSession\
        .builder\
        .appName("PythonWordCountWithParallelize")\
        .getOrCreate()


    path = 'shahname.txt'
    with open(path, 'r') as file:
        lines = file.readlines()

    rdd = spark.sparkContext.parallelize(lines)
    final_lines = rdd.filter(lambda line: not line.strip().startswith('#') and line.strip() != "")

    line_counts = final_lines.map(lambda line: 1).reduce(add)
    print(f'Total lines: {line_counts}')

    words = final_lines.flatMap(lambda line: line.split())
    word_counts = words.map(lambda word: 1).reduce(add)
    print(f'Total words: {word_counts}')

    distinct_words = words.distinct()
    num_distinct_words = distinct_words.map(lambda word: 1).reduce(add)
    print(f'Total distinct words: {num_distinct_words}')

    spark.stop()

endTime = time.time()
totalTime = endTime - startTime
print(f"Total time: {totalTime:} seconds")

Total lines: 22845
Total words: 261190
Total distinct words: 12535
Total time: 6.931774139404297 seconds


In [ ]:
#1 - ب
import sys
from operator import add
from pyspark.sql import SparkSession
import time
startTime = time.time()

spark = SparkSession\
  .builder \
  .appName("PythonWordCount") \
  .getOrCreate()

path = 'shahname.txt'

lines = spark.read.text(path).rdd.map(lambda r: r[0])
#ابیات
final_lines = lines.filter(lambda line: not line.strip().startswith('#') and line.strip() != "")
split_lines = final_lines.flatMap(lambda line: line.split("\t"))

outputMesra = 'output.txt'
with open(outputMesra , 'w') as f:
  for i in split_lines.collect():
    f.write(i + '\n')

outputGafie = 'output_gafie.txt'
gafieh = spark.read.text(outputMesra).rdd.map(lambda line: line[0].split()[-1] if line[0] else '')
with open(outputGafie, 'w') as g:
  for i in gafieh.collect():
    g.write(i + '\n')

GafiehHa = spark.read.text('output_gafie.txt')
counts = GafiehHa.rdd.flatMap(lambda x: x[0].split(' ')) \
                  .map(lambda x: (x, 1)) \
                  .reduceByKey(add)
NewOutput = counts.collect()

repeats = []
for (word, count) in NewOutput:
  repeats.append({'count' : count , 'word' : word })

sortedRepeatsOrderCount = sorted(repeats,key = lambda item : item['count'] , reverse=True)

for i in range(10):
  print(f'top {i+1} is : {sortedRepeatsOrderCount[i]}')
endTime = time.time()
totalTime = endTime - startTime
print(f"Total time: {totalTime:} seconds")

top 1 is : {'count': 1131, 'word': 'بود'}
top 2 is : {'count': 1130, 'word': 'شاه'}
top 3 is : {'count': 1046, 'word': 'سپاه'}
top 4 is : {'count': 700, 'word': 'اوی'}
top 5 is : {'count': 584, 'word': 'شهریار'}
top 6 is : {'count': 559, 'word': 'کرد'}
top 7 is : {'count': 556, 'word': 'راه'}
top 8 is : {'count': 536, 'word': 'را'}
top 9 is : {'count': 511, 'word': 'شد'}
top 10 is : {'count': 466, 'word': 'من'}
Total time: 10.53723692893982 seconds


In [ ]:
# 1- ج
from pyspark.sql import SparkSession
from pyspark.ml.feature import NGram
from pyspark.sql.functions import col, split, monotonically_increasing_id, explode
import time
startTime = time.time()

if __name__ == "__main__":
    spark = SparkSession.builder.appName("NGramExample").getOrCreate()

    df = spark.read.text("output.txt")

    wordDataFrame = df.withColumn("words", split(col("value"), " ")) \
                      .withColumn("id", monotonically_increasing_id()) \
                      .select("id", "words")

    ngram = NGram(n=3, inputCol="words", outputCol="ngrams")
    ngramDataFrame = ngram.transform(wordDataFrame)

    ngramFlattened = ngramDataFrame.withColumn("ngram", explode("ngrams"))
    ngramCount = ngramFlattened.groupBy("ngram").count().orderBy("count", ascending=False)

    ngramList = ngramCount.collect()

    for i in range(10):
        print(f"top {i + 1} : ' {ngramList[i]['ngram']} ' with  count {ngramList[i]['count']}")

    spark.stop()
endTime = time.time()
totalTime = endTime - startTime
print(f"Total time: {totalTime:} seconds")

top 1 : ' چنین داد پاسخ ' with  count 306
top 2 : ' داد پاسخ که ' with  count 231
top 3 : ' چنین گفت کای ' with  count 134
top 4 : ' مر او را ' with  count 126
top 5 : ' چنین گفت با ' with  count 124
top 6 : ' تاج و تخت ' with  count 109
top 7 : ' بدو گفت کای ' with  count 108
top 8 : ' ز هر سو ' with  count 92
top 9 : ' نشست از بر ' with  count 85
top 10 : ' تو گفتی که ' with  count 70
Total time: 18.562411069869995 seconds
